In [ ]:
# pip install --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

In [5]:
pip install --upgrade transformers sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [2]:
chunks = [
    {
        "id": "chunk_001",
        "text": "Python is a programming language used for machine learning.",
        "topic": "Python"
    },
    {
        "id": "chunk_002",
        "text": "PyTorch is a framework used to build and train deep learning models.",
        "topic": "PyTorch"
    },
    {
        "id": "chunk_003",
        "text": "Neo4j stores data as nodes and relationships in a graph.",
        "topic": "Neo4j"
    }
] 

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2'
)

for chunk in chunks:
    vector = model.encode(
        chunk["text"],
        normalize_embeddings=True
    )

    chunk["embedding"] = vector.tolist()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [21]:
from neo4j import GraphDatabase

# Neo4j Desktop default URI is bolt://localhost:7687
URI = "bolt://localhost:7687"
AUTH = ("neo4j", "12345678")  # Replace with your actual password

driver = GraphDatabase.driver(URI, auth=AUTH)

In [22]:
with driver.session() as session:
    session.run("""
    CREATE VECTOR INDEX chunk_embedding_index IF NOT EXISTS
    FOR (c:Chunk) ON (c.embedding)
    OPTIONS {
      indexConfig: {
        `vector.dimensions`: 384,
        `vector.similarity_function`: 'cosine'
      }
    }
    """)
print("Vector index created successfully!")

Vector index created successfully!


In [23]:
# use unwind for batch ingestion 
def insert_chunk(tx,chunks):
    result = tx.run(
        """
        UNWIND $chunks AS item
        MERGE (c:Chunk{id:item.id})
        SET c.text = item.text,
        c.topic = item.topic,
        c.embedding = item.embedding
        RETURN count(c) AS processed
        """,
        chunks = chunks
    )
    return result.single()['processed']

In [24]:
with driver.session() as session:
    count = session.execute_write(
        insert_chunk,
        chunks
    )
    print("Processed:",count)

Processed: 3


In [25]:
## 9.8 # query the vector index
question = "which tool help build deep learning models ?"

query_vector = model.encode(
    question,
    normalize_embeddings=True
).tolist()

In [26]:
def vector_search(tx,query_vector,top_k):
    result = tx.run(
        """
        CALL db.index.vector.queryNodes(
            'chunk_embedding_index',
            $top_k,
            $query_vector
        )
        YIELD node, score
        RETURN 
            node.id as id,
            node.text as text,
            node.topic as topic,
            score
        ORDER BY score DESC
        """,
        query_vector = query_vector,
        top_k = top_k
    )
    return [record.data() for record in result]



In [27]:
with driver.session() as session:
    results = session.execute_read(
        vector_search,query_vector,3
    )
    for item in results:
        print(item['topic'],item['score'])
        print(item['text'])

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=9, offset=9>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 9, 'line': 2, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        CALL db.index.vector.queryNodes(\n            'chunk_embedding_index',\n            $top_k,\n            $query_vector\n        )\n        YIELD node, score\n        RETURN \n            node.id as id,\n            node.text as text,\n            node.topic as topic,\n            score\n        ORDER BY score DESC\n        "


PyTorch 0.8287009000778198
PyTorch is a framework used to build and train deep learning models.
Python 0.6203766465187073
Python is a programming language used for machine learning.
Neo4j 0.5811343193054199
Neo4j stores data as nodes and relationships in a graph.


The new error in cell `[19]` is:

```text
ClientError: Failed to invoke procedure `db.index.vector.queryNodes`: 
Caused by: java.lang.IllegalArgumentException: There is no such vector schema index: chunk_embedding_index

```

### Why This Happened

Your `vector_search` function is querying a Neo4j vector index named `'chunk_embedding_index'`, but that index hasn't been created in your active Neo4j database yet (or was named differently).

---

### How to Fix It

Run a cell in your notebook to create the index before querying it.

**1. Create the Vector Index**
Execute this code block once in a cell to define the index on your `Chunk` nodes (adjust `embedding` property name and vector dimensions—usually `384` or `768`—if needed):

```python
with driver.session() as session:
    session.run("""
    CREATE VECTOR INDEX chunk_embedding_index IF NOT EXISTS
    FOR (c:Chunk) ON (c.embedding)
    OPTIONS {
      indexConfig: {
        `vector.dimensions`: 384,
        `vector.similarity_function`: 'cosine'
      }
    }
    """)
print("Vector index created successfully!")

```

**2. Verify Existing Indexes (Optional)**
If you already created an index under a different name, run this snippet to see all index names in your database:

```python
with driver.session() as session:
    result = session.run("SHOW INDEXES YIELD name, type WHERE type = 'VECTOR'")
    for record in result:
        print(record["name"])

```

**3. Re-run Cell `[19]**`
Once `chunk_embedding_index` is created, executing cell `[19]` will run the vector search query without failing.